# K-Nearest Neighbors

This notebook classifies each *region* into the cluster found by k-means, using the k nearest regions.
To check how well it works, we use **k-fold cross validation**.

In [1]:
import importlib
from pathlib import Path

import models.knearest as knearest
importlib.reload(knearest)   # always use the latest saved knearest.py
from models.knearest import *

### Settings

In [2]:
clusters = 5     # number of k-means clusters (the labels)
folds = 5        # number of groups for cross validation
k = 3            # number of neighbors
p = 2            # distance: 1 = Manhattan, 2 = Euclidean

### Step 1: Read, clean up and standardize the dataset, then get the clusters

In [3]:
from pathlib import Path

current_dir_parent = Path.cwd().resolve()
data_set_dir = current_dir_parent / "PROJECT_DATASET"
regions, featuresList, labelList = prepareData(data_set_dir, clusters)

print(len(regions), "regions,", len(featuresList[0]), "features")
print("Clusters:", labelList)

Initial Centroids
(-0.8379421076536361, -0.6148966074056751, -0.3243437160251675, -1.0014745088255963, -1.245454230901362, -0.39315584434697365, -0.1304891202488182, -0.3187963026966897, 0.4326464897807249, 1.7696772521100204, 0.021215457697741174, 0.9642981531085223, -0.5781659696158682, -0.28169334234730836, 0.6779117516120259, -0.4332684085305676, -0.5476365379234204, -0.4361785701069968, -0.4586095814696183)
(-0.21714764346199303, -0.4619756120425082, -0.1145779431610645, -0.09588585722798247, -0.146355756165454, -0.6394422140569772, -0.008882082648275134, -0.5042090895366287, -0.8174111546054804, -0.2664609094431817, -0.47469586598695135, -0.5969464757338468, -0.9421963949295631, -0.481226126509985, -0.28246322983834404, -0.20309456649870344, 0.644474706214757, -0.05799824338399049, 0.9148429474912051)
(-1.081825647157496, -1.0518137370147231, -1.0210657473237952, -1.0014745088255963, 0.2388025837059529, 2.054290865603766, 0.8060298395582237, 1.3052433090243911, 0.5489309218166506

### Step 2: Shuffle once and store the folds

In [4]:
foldList = kFoldSplit(len(featuresList), folds)
printFolds(regions, foldList)

Fold 1: CAR, Region X, Region II, Region V
Fold 2: Region VI, Region IV-A, Region XI, BARMM
Fold 3: Region XIII, Region III, Region I
Fold 4: MIMAROPA, Region VIII, Region IX
Fold 5: NCR, Region VII, Region XII


### Step 3: K-nearest neighbor with cross validation
`/` means the predicted cluster is the same as the k-means cluster, `x` means it is not.

In [5]:
results = crossValidation(featuresList, labelList, foldList, k, p)
printOutput(k, p, regions, labelList, results)
printAccuracy(results)

K-Nearest Neighbor Output from the regions
k: 3	p: 2


Region       Fold  K-means  KNN      Nearest regions (cluster)
NCR          5     4        4     /  Region IV-A (4), CAR (0), Region III (2)
CAR          1     0        2     x  Region VI (2), Region III (2), Region I (2)
Region I     3     2        0     x  Region VI (2), Region II (0), Region VII (0)
Region II    1     0        2     x  Region VI (2), Region VII (0), Region III (2)
Region III   3     2        0     x  Region VI (2), Region II (0), Region VII (0)
Region IV-A  2     4        4     /  NCR (4), CAR (0), Region III (2)
MIMAROPA     4     1        1     /  Region XI (1), Region XII (3), Region VI (2)
Region V     1     3        3     /  BARMM (3), Region VIII (3), Region XII (3)
Region VI    2     2        2     /  Region III (2), Region I (2), Region II (0)
Region VII   5     0        0     /  Region II (0), Region VI (2), Region X (3)
Region VIII  4     3        3     /  Region V (3), BARMM (3), Region X (3)
Region I